## Cleaning the raw conversations

In [1]:
import json
import re
from pathlib import Path

raw_cognitive_path = Path("LLM Cognitive Thinking.txt")
raw_behaviour_path = Path("Usercase_HeartDisease.txt")

#making sure the files are loaded 
print("Cognitive log found:", raw_cognitive_path.exists())
print("Behaviour log found:", raw_behaviour_path.exists())


Cognitive log found: True
Behaviour log found: True


In [5]:

def clean_text(s):

    if not isinstance(s, str):
        return ""
    s = s.replace("\ufeff", "")   
    return s.strip()


In [7]:

#raw cognitive log into nodes now

def parse_cognitive_log(path):
    
    raw_text = Path(path).read_text(encoding="utf-8")

   
    blocks = re.findall(r"\{.*?\}", raw_text, flags=re.DOTALL)

    cleaned_nodes = []

    for block in blocks:
        
        try:
            obj = json.loads(block)

        except json.JSONDecodeError:
            
            repaired = block.replace("'", '"')
            try:
                obj = json.loads(repaired)
            except:
                # Skip if still unparseable
                continue

       
        for key in ["summary", "explanation", "next_step", "raw_message"]:
            if key in obj:
                obj[key] = clean_text(obj[key])

        # Clean reasoning steps only structurally, not semantically
        if "reasoning_steps" in obj and isinstance(obj["reasoning_steps"], list):
            obj["reasoning_steps"] = [clean_text(x) for x in obj["reasoning_steps"]]

        cleaned_nodes.append(obj)

    return cleaned_nodes



cognitive_nodes = parse_cognitive_log(raw_cognitive_path)

#check how many nodes were parsed
print("Cognitive nodes loaded:", len(cognitive_nodes))
if len(cognitive_nodes) > 0:
    print("First node preview:", list(cognitive_nodes[0].keys()))


Cognitive nodes loaded: 30
First node preview: ['id', 'speaker', 'summary', 'reasoning_steps', 'explanation', 'next_step', 'raw_message', 'parent']


In [9]:
#now edges 
def build_cognitive_edges(nodes):

    edges = []

    for i in range(len(nodes) - 1):
        src = nodes[i].get("id")
        tgt = nodes[i + 1].get("id")

        if src and tgt:
            edges.append({"source": src, "target": tgt})

    return edges

cognitive_edges = build_cognitive_edges(cognitive_nodes)

print("Cognitive edges created:", len(cognitive_edges))
if len(cognitive_edges) > 0:
    print("First edge:", cognitive_edges[0])


Cognitive edges created: 29
First edge: {'source': 'H0', 'target': 'L0'}


In [11]:
#behaviour conversation
def parse_behaviour_log(path):
   
    raw_text = Path(path).read_text(encoding="utf-8")

    blocks = re.findall(r"\{.*?\}", raw_text, flags=re.DOTALL)

    cleaned_nodes = []

    for block in blocks:
        try:
            obj = json.loads(block)
        except json.JSONDecodeError:
            
            repaired = block.replace("'", '"')
            try:
                obj = json.loads(repaired)
            except:
                continue  

        if "user_message" in obj:
            obj["user_message"] = clean_text(obj["user_message"])

        if "assistant_response" in obj:
            obj["assistant_response"] = clean_text(obj["assistant_response"])

        if "tags" in obj:
           
            if isinstance(obj["tags"], list):
                obj["tags"] = [clean_text(t) for t in obj["tags"]]

        if "phase" in obj:
            obj["phase"] = clean_text(obj["phase"])

        cleaned_nodes.append(obj)

    return cleaned_nodes


#behaviour nodes
behaviour_nodes = parse_behaviour_log(raw_behaviour_path)

print("Behavioural nodes loaded:", len(behaviour_nodes))
if len(behaviour_nodes) > 0:
    print("First behavioural node keys:", list(behaviour_nodes[0].keys()))


Behavioural nodes loaded: 11
First behavioural node keys: ['user_message', 'assistant_response', 'type', 'tags', 'phase']


In [16]:
#behaviour edges 
def build_behaviour_edges(nodes):
    """
    Create simple sequential edges:
        i -> i+1
    Where each node is referenced by its index in the list.
    """
    edges = []
    
    for i in range(len(nodes) - 1):
        edges.append({"source": i, "target": i + 1})
    
    return edges

behaviour_edges = build_behaviour_edges(behaviour_nodes)

print("Behaviour edges created:", len(behaviour_edges))
if len(behaviour_edges) > 0:
    print("First behaviour edge:", behaviour_edges[0])


Behaviour edges created: 10
First behaviour edge: {'source': 0, 'target': 1}


In [18]:
# -------------------------------------------------------------------
# Save the reconstructed cognitive nodes as cognitive_nodes_clean.json
# -------------------------------------------------------------------

output_path_nodes = Path("cognitive_nodes_clean2.json")

with output_path_nodes.open("w", encoding="utf-8") as f:
    json.dump(cognitive_nodes, f, indent=2, ensure_ascii=False)

print("Saved:", output_path_nodes.resolve())


Saved: C:\Users\Rakeen Afser\cognitive_nodes_clean2.json
